## Ribbon Load Balancer
Components of Ribbon Load Balancer are listed below:

### ILoadBalancer
Is defined as:

In [ ]:
public interface ILoadBalancer {

    // Initial list of servers + use this method to add servers later    
    public void addServers(List<Server> newServers);

    // Choose a server from the available options using key
    public Server chooseServer(Object key);

    public void markServerDown(Server server);

    // Return list of servers that are up and reachable
    public List<Server> getReachableServers();

    // Return list of all servers    
    public List<Server> getAllServers();
    
    // ... deprecated method
}

public class Server {
    private String host;
    private int port;
    
    private volatile String id;
    private volatile boolean isAliveFlag;
    private volatile boolean readyToServe = true;
    
    // other properties of the server
}

`AbstractLoadBalancer` implements `ILoadBalancer` and defines the buckets under which servers may be grouped. It also uses null key to choose a server: 

In [ ]:
public abstract class AbstractLoadBalancer implements ILoadBalancer {
    // Bucket definition
    public enum ServerGroup{
        ALL,
        STATUS_UP,
        STATUS_NOT_UP        
    }
        
    public Server chooseServer() {
        return chooseServer(null);
    }

    // Return list of servers belonging to the given bucket    
    public abstract List<Server> getServerList(ServerGroup serverGroup);
    
    // other method
}

`BaseLoadBalancer` is a concrete implementation which uses `IRule`, `IPing` and `IPingStrategy` to load balance:

In [ ]:
public class BaseLoadBalancer implements AbstractLoadBalancer implements PrimeConnections.PrimeConnectionListener, IClientConfigAware {
    protected IRule rule = DEFAULT_RULE; // Default is RoundRobinRule
    protected IPingStrategy pingStrategy = DEFAULT_PING_STRATEGY;  // Default is SerialPingStrategy
    protected IPing ping = null;

    protected volatile List<Server> allServerList = Collections.synchronizedList(new ArrayList<Server>());
    protected ReadWriteLock allServerLock = new ReentrantReadWriteLock();
    
    protected volatile List<Server> upServerList = Collections.synchronizedList(new ArrayList<Server>());
    protected ReadWriteLock upServerLock = new ReentrantReadWriteLock();
    
    @Override
    public void addServers(List<Server> newServers) {
        if (newServers != null && newServers.size() > 0) {
            try {
                ArrayList<Server> newList = new ArrayList<>();
                newList.addAll(allServerList);
                newList.addAll(newServers);
                setServersList(newList);
            } catch (Exception e) {
                logger.error("LoadBalancer [{}]: Exception while adding Servers", name, e);
            }
        }
    }
    
    // Approximate implementation, differs from the actual implementation
    public void setServersList(List<Server> servers) {
        Lock writeLock = allServerLock.writeLock();
        
        writeLock.lock();
        try {
            if (!allServerList.equals(allServers)) {
                listChanged = true;
            }
            
            allServerList = servers;
            
            if(listChanged) {
                forceQuickPing();  // original list of servers has changed, ping to determine which ones are up
            }
            
        } finally {
            writeLock.unlock();
        }
    }
    
    public void forceQuickPing() {
        if (ping == null || ping.getClass().equals(DummyPing.class)) {
            return;
        }
        logger.debug("LoadBalancer [{}]:  forceQuickPing invoking", name);
        
        try {
            new Pinger(pingStrategy).runPinger();
        } catch (Exception e) {
            logger.error("LoadBalancer [{}]: Error running forceQuickPing()", name, e);
        }
    }
    
    @Override
    public Server chooseServer(Object key) {
        if(rule == null) return null;
        
        try {
            return rule.choose(key);
        } catch (Exception e) {
            logger.warn("LoadBalancer [{}]:  Error choosing server for key {}", name, key, e);
            return null;
        }
    }
    
    @Override
    public List<Server> getReachableServers() {
        return Collections.unmodifiableList(upServerList);
    }
    
    // Contains a PingTask which runs every x seconds and uses Pinger
}

A `DynamicServerListLoadBalancer` extends `BaseLoadBalancer` and uses a `ServerListUpdater` and `ServerList` to fetch list of servers:

In [ ]:
// This will be used by ServerListUpdater
public interface ServerList<T extends Server> {
    public List<T> getInitialListOfServers();
    
    // Returns list of servers
    public List<T> getUpdatedListOfServers();   
}

In [ ]:
public class DynamicServerListLoadBalancer<T extends Server> extends BaseLoadBalancer {
    volatile ServerList<T> serverListImpl;
    protected volatile ServerListUpdater serverListUpdater;  // contains a ScheduledTaskExecutor
    protected AtomicBoolean serverListUpdateInProgress = new AtomicBoolean(false);
    
    public void updateListOfServers() {
        List<T> servers = new ArrayList<T>();
        if (serverListImpl != null) {
            servers = serverListImpl.getUpdatedListOfServers();
            LOGGER.debug("List of Servers for {} obtained from Discovery client: {}",
                    getIdentifier(), servers);
        }
        
        updateAllServerList(servers);
    }
    
    protected void updateAllServerList(List<T> ls) {
        // other threads might be doing this - in which case, we pass
        if (serverListUpdateInProgress.compareAndSet(false, true)) {
            try {
                for (T s : ls) {
                    s.setAlive(true); // set so that clients can start using these
                                      // servers right away instead
                                      // of having to wait out the ping cycle.
                }
                setServersList(ls);
                super.forceQuickPing();
            } finally {
                serverListUpdateInProgress.set(false);
            }
        }
    }
    
    @Override
    public void forceQuickPing() { /* no-op */ }
}

### IPing and IPingStrategy
The `IPing` interface defines how a server is pinged:

In [ ]:
public interface IPing {
    // checks if server is alive and can be used for load balancing
    public boolean isAlive(Server server);
}

`IPingStrategy` defines how all servers should be pinged using `IPing`, for example sequentially or parallely:

In [ ]:
public interface IPingStrategy {
    boolean[] pingServers(IPing ping, Server[] servers);
}

`BaseLoadBalancer` defines a class `Pinger` to ping all the servers:

In [ ]:
// Simplified implementation
class Pinger {
    public void runPinger() throws Exception {
        Server[] allServers = null;
        boolean[] results = null;

        // transform all servers list to array as required by IPingStrategy
        Lock allLock = allServerLock.readLock();
        allLock.lock();
        allServers = allServerList.toArray(new Server[allServerList.size()]);
        allLock.unlock();

        results = pingerStrategy.pingServers(ping, allServers);

        final List<Server> newUpList = new ArrayList<Server>();

        for (int i = 0; i < allServers.length; i++) {
            boolean isAlive = results[i];
            Server svr = allServers[i];

            svr.setAlive(isAlive);

            if (isAlive) {
                newUpList.add(svr);
            }
        }

        // Replace list of servers available with this new list
        Lock upLock = upServerLock.writeLock();
        upLock.lock();
        upServerList = newUpList;
        upLock.unlock();
}

### IRule
Contains a reference to the load balancer and picks a server from all servers or up servers:

In [ ]:
public interface IRule{
    public Server choose(Object key);    
    public void setLoadBalancer(ILoadBalancer lb);    
    public ILoadBalancer getLoadBalancer();    
}

The most notable implementation is `RoundRobinRule`:

In [ ]:
public class RoundRobinRule extends AbstractLoadBalancerRule {
    private AtomicInteger nextServerCyclicCounter = new AtomicInteger(0);
    
    @Override
    public Server choose(Object key) {
        Server server = null;
        int count = 0;
        
        while (server == null && count++ < 10) {
            List<Server> reachableServers = lb.getReachableServers();
            List<Server> allServers = lb.getAllServers();

            if ((reachableServers.size() == 0) || (allServers.size() == 0)) {
                log.warn("No up servers available from load balancer: " + lb);
                return null;
            }

            int nextServerIndex = incrementAndGetModulo(allServers.size());
            server = allServers.get(nextServerIndex);

            if (server == null) {
                /* Transient. */
                Thread.yield();
                continue;
            }

            if (server.isAlive() && (server.isReadyToServe())) {
                return (server);
            }

            // Next.
            server = null;
        }

        if (count >= 10) log.warn("No available alive servers after 10 tries from load balancer: " + lb);
        
        return server;
    }
    
    private int incrementAndGetModulo(int modulo) {
        for (;;) {
            int current = nextServerCyclicCounter.get();
            int next = (current + 1) % modulo;
            if (nextServerCyclicCounter.compareAndSet(current, next))
                return next;
        }
    }
}

## Spring Cloud Load Balancer
Ribbon went into maintenance mode around 2016. Since then Spring team has released its own implementation of load balancer as Spring Cloud LoadBalancer.